In [1]:
from pymol import cmd
import numpy as np
import os

def remove_first_ter_after_chain_b(input_pdb, output_pdb):
    chain_b_started = False
    ter_skipped = False

    with open(input_pdb, 'r') as infile, open(output_pdb, 'w') as outfile:
        for line in infile:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                chain_id = line[21].strip()  # Column 5 (21st character in 0-indexed line)
                if chain_id == "B":
                    chain_b_started = True

            if chain_b_started and line.startswith("TER") and not ter_skipped:
                ter_skipped = True  # Skip the first TER after chain B starts
                continue

            outfile.write(line)  # Write all other lines

def main(cif_file):
    cmd.reinitialize()
    cmd.load(cif_file, 'PIK3CB_guide_target') 
    cmd.load('4f3t_v4G2c.pdb')
    
    cmd.select('chainB_resi3_4', 'chain B and (resi 3 or resi 4)')
    cmd.select('chainR_resi2_3', 'chain R and (resi 2 or resi 3)')
    cmd.create('new_obj_chainR', 'chainR_resi2_3')
    cmd.align('new_obj_chainR', 'chainB_resi3_4')
    cmd.select('chainB_atoms', 'chain B and ((resi 3 and name O3\') or (resi 4 and (name OP1 or name OP2 or name P or name C5\' or name O5\')))')
    cmd.select('keep_new_obj_chainR_atoms', 'new_obj_chainR and ((resi 2 and (name C or name O or name C6\')) or (resi 3 and (name N or name C5\')))')
    cmd.remove('new_obj_chainR and not keep_new_obj_chainR_atoms')
    cmd.remove('chainB_atoms')
    cmd.create('PIK3CB_guide_target', 'PIK3CB_guide_target or new_obj_chainR')
    cmd.select('atom1', 'PIK3CB_guide_target and chain B and resi 3 and name C3\'')
    cmd.select('atom2', 'PIK3CB_guide_target and chain R and name C6\'')
    cmd.bond('atom1', 'atom2')
    cmd.select('atom3', 'PIK3CB_guide_target and chain B and resi 4 and name C4\'')
    cmd.select('atom4', 'PIK3CB_guide_target and chain R and name C5\'')
    cmd.bond('atom3', 'atom4')

    distance_atom3_atom4 = cmd.get_distance('atom3', 'atom4')
    coord_atom3 = np.array(cmd.get_atom_coords('atom3'))
    coord_atom4 = np.array(cmd.get_atom_coords('atom4'))
    vector3_4 = coord_atom4 - coord_atom3
    vector3_4_normalized = vector3_4 / np.linalg.norm(vector3_4)
    current_distance_3_4 = distance_atom3_atom4
    desired_distance_3_4 = 1.6
    translation_vector_3_4 = vector3_4_normalized * (current_distance_3_4 - desired_distance_3_4)
    if current_distance_3_4 > desired_distance_3_4:
        cmd.translate(list(-translation_vector_3_4), 'atom4')
    
    distance_atom1_atom2 = cmd.get_distance('atom1', 'atom2')
    coord_atom1 = np.array(cmd.get_atom_coords('atom1'))
    coord_atom2 = np.array(cmd.get_atom_coords('atom2'))
    vector1_2 = coord_atom2 - coord_atom1
    vector1_2_normalized = vector1_2 / np.linalg.norm(vector1_2)
    current_distance_1_2 = distance_atom1_atom2
    desired_distance_1_2 = 1.6
    translation_vector_1_2 = vector1_2_normalized * (current_distance_1_2 - desired_distance_1_2)
    if current_distance_1_2 > desired_distance_1_2:
        cmd.translate(list(-translation_vector_1_2), 'atom2')
    
    cmd.alter('atom2', 'chain="B"')
    cmd.alter('atom2', 'resi=3')
    cmd.alter('atom2', 'resn="A"')
    cmd.alter('atom2', 'segi="B"')
    cmd.alter('chain R', 'resi=4')
    cmd.alter('chain R', 'resn="G"')
    cmd.alter('chain R', 'segi="B"')
    cmd.alter('chain R', 'chain="B"')
    cmd.alter("chain B and resi 3", "resn='R3'")
    cmd.alter("chain B and resi 4", "resn='R4'")
    ###########
    cmd.sort()
    
    cmd.select("phosphate_atoms", "chain B and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")
    cmd.select("phosphate_atoms", "chain C and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")

    ##R3
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C5'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H02'")
    cmd.attach("H", 1, 1)
    cmd.unpick()
    cmd.select("incorrect_hydrogen", "(neighbor target_atom) and elem H and not name H02")
    cmd.alter("incorrect_hydrogen", "name='H06'")
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/N6")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H11'")
    cmd.attach("H", 1, 1)
    cmd.unpick()
    cmd.select("incorrect_hydrogen", "(neighbor target_atom) and elem H and not name H11")
    cmd.alter("incorrect_hydrogen", "name='H12'")
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C6'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H01'")
    cmd.attach("H", 1, 1)
    cmd.unpick()
    cmd.select("incorrect_hydrogen", "(neighbor target_atom) and elem H and not name H01")
    cmd.alter("incorrect_hydrogen", "name='H03'")
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C4'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H04'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C1'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H09'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C8")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H13'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C2")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H10'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C3'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H05'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/C2'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H07'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/3/O2'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H08'")
    cmd.unpick()
    
    ########################### R4
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/N2")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H08'")
    cmd.attach("H", 1, 1)
    cmd.unpick()
    cmd.select("incorrect_hydrogen", "(neighbor target_atom) and elem H and not name H08")
    cmd.alter("incorrect_hydrogen", "name='H09'")
    
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C5'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H01'")
    cmd.attach("H", 1, 1)
    cmd.unpick()
    cmd.select("incorrect_hydrogen", "(neighbor target_atom) and elem H and not name H01")
    cmd.alter("incorrect_hydrogen", "name='H02'")
    
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C4'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H03'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C3'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H04'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C2'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H05'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/O2'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H06'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C1'")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H07'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/C8")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H10'")
    cmd.unpick()
    
    cmd.select("target_atom", "/PIK3CB_guide_target//B/4/N")
    cmd.edit("target_atom")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor target_atom)", "name='H14'")
    cmd.unpick()

    
    cmd.sort()

    

    
    cmd.save('PIK3CB_guide_target_amide_TER.pdb', 'PIK3CB_guide_target')
    
    input_pdb = "PIK3CB_guide_target_amide_TER.pdb"
    remove_first_ter_after_chain_b(input_pdb, "pdb_with_modification.pdb")

cif_file = "ENSG00000174780.cif"
main(cif_file)

In [68]:
## remove phosphates
from pymol import cmd
import numpy as np
import os

def main(cif_file):
    cmd.reinitialize()
    cmd.load(cif_file, 'PIK3CB_guide_target') 
    cmd.select("phosphate_atoms", "chain B and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")
    cmd.select("phosphate_atoms", "chain C and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")
    cmd.save('pdb_with_modification.pdb', 'PIK3CB_guide_target')
cif_file = "ENSG00000051382.cif"
# cif_file = "ENSG00000001497.cif"

main(cif_file)

In [ ]:
### MODIFIES MDP FILE
import subprocess
import os
import shutil

os.environ['PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/bin:' + os.environ['PATH']
if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib:' + os.environ['LD_LIBRARY_PATH']
else:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib'
os.environ['GMX_MAXBACKUP'] = '-1'

def run_command(command, input_text=None, max_chars=100):
    result = subprocess.run(command, capture_output=True, text=True, input=input_text)
    output = result.stdout + result.stderr
    for line in output.splitlines():
        if "warning" in line.lower() or "fatal" in line.lower():
            print(line[:max_chars])

equi_prefix = "step4.0_minimization"
nsteps = 5000

for i in range(1, 11):
    with open(f"{equi_prefix}.mdp", 'r') as file:
        mdp_content = file.readlines()

    with open(f"{equi_prefix}_temp.mdp", 'w') as file:
        for line in mdp_content:
            if line.startswith("nsteps"):
                line = f"nsteps = {nsteps}\n"
            file.write(line)

    command = ["gmx", "grompp", "-v", "-f", "step4.0_minimization_temp.mdp", "-o", "step4.0_minimization.tpr", 
               "-c", "structure_solv_ions.gro", "-r", "structure_solv_ions.gro", "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5"]
    run_command(command)

    command = ["gmx", "mdrun", "-v", "-deffnm", "step4.0_minimization", "-ntmpi", "1"]
    run_command(command)

    mv_command = ["mv", "step4.0_minimization.gro", f"step4.0_minimization_{i}.gro"]
    run_command(mv_command)
    nsteps += 10000
%run graphs_mini.py


In [ ]:
## All steps + repeats
import subprocess
import os
import shutil

os.environ['PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/bin:' + os.environ['PATH']
if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib:' + os.environ['LD_LIBRARY_PATH']
else:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib'
os.environ['GMX_MAXBACKUP'] = '-1'

def run_command(command, input_text=None, max_chars=100):
    result = subprocess.run(command, capture_output=True, text=True, input=input_text)
    output = result.stdout + result.stderr
    for line in output.splitlines():
        if "warning" in line.lower() or "fatal" in line.lower():
            print(line[:max_chars])

def run_gromacs_commands():
    command = ["gmx", "pdb2gmx", "-f", "pdb_with_modification.pdb", "-o", "structure_processed.gro", 
               "-p", "topol.top", "-i", "posre.itp"]
    input_text = "6\n1\n"
    run_command(command, input_text)

    command = ["gmx", "editconf", "-f", "structure_processed.gro", "-o", "structure_box.gro", "-c", "-d", "1.0", "-bt", "cubic"]
    run_command(command)

    command = ["gmx", "solvate", "-cp", "structure_box.gro", "-cs", "spc216.gro", "-o", "structure_solv.gro", "-p", "topol.top"]
    run_command(command)

    command = ["gmx", "grompp", "-f", "ions.mdp", "-c", "structure_solv.gro", "-p", "topol.top", "-o", "ions.tpr", "-maxwarn", "3"]
    run_command(command)

    command = ["gmx", "genion", "-s", "ions.tpr", "-o", "structure_solv_ions.gro", "-p", "topol.top", 
               "-pname", "NA", "-nname", "CL", "-neutral", "-conc", "0.15"]
    input_text = "14\n"
    run_command(command, input_text)

    command = ["gmx", "make_ndx", "-f", "structure_solv_ions.gro", "-o", "index.ndx"]
    input_text = "name 19 SOLV\n1 | 12\nname 20 SOLU\nq\n"
    run_command(command, input_text)

    command = ["gmx", "grompp", "-v", "-f", "step4.0_minimization.mdp", "-o", "step4.0_minimization.tpr", 
               "-c", "structure_solv_ions.gro", "-r", "structure_solv_ions.gro", "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5"]
    run_command(command)

    command = ["gmx", "mdrun", "-v", "-deffnm", "step4.0_minimization", "-ntmpi", "1"]
    run_command(command)

    equi_prefix = "step4.0_minimization"
    prod_prefix = "step5_production"
    prod_step = "step5"

    grompp_command = [
        "gmx", "grompp", "-f", f"{prod_prefix}.mdp", "-o", f"{prod_step}.tpr",
        "-c", f"{equi_prefix}.gro", "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "2"
    ]
    run_command(grompp_command)

    mdrun_command = ["gmx", "mdrun", "-v", "-deffnm", prod_step, "-ntmpi", "1"]
    run_command(mdrun_command)

    mv_command = ["mv", "step5.gro", f"step5_{i}.gro"]
    run_command(mv_command)

# Directory for output files
output_directory = "minimization_gros4"
os.makedirs(output_directory, exist_ok=True)

# Names of the resulting .gro files
gro_files = [
    # "structure_processed.gro",
    # "structure_box.gro",
    # "structure_solv.gro",
    # "structure_solv_ions.gro",
    # "step4.0_minimization.gro"
    "step5.gro"
]

for i in range(1, 11):  # Loop 10 times
    run_gromacs_commands()
    
    for gro_file in gro_files:
        if os.path.exists(gro_file):
            # Create new file name for this iteration
            base_name = os.path.splitext(gro_file)[0]
            new_file_name = f"{base_name}_{i}.gro"
            new_file_path = os.path.join(output_directory, new_file_name)
            shutil.move(gro_file, new_file_path)
            print(f"Moved '{gro_file}' to '{new_file_path}'")

In [23]:
import os

dir1_amid = '/data/home/mrichte3/RNASeq/amide/step5'
dir2_gna = '/data/home/mrichte3/RNASeq/gna/step5'

files_dir1 = set(os.listdir(dir1_amid))
files_dir2 = set(os.listdir(dir2_gna))

print(f'Total files in {dir1_amid}: {len(files_dir1)}')
print(f'Total files in {dir2_gna}: {len(files_dir2)}')

common_files = files_dir1.intersection(files_dir2)
common_file_count = len(common_files)
print(f'Number of common files: {common_file_count}')

unique_files_dir1 = files_dir1 - common_files
unique_files_dir2 = files_dir2 - common_files

print(f'Number of unique files in {dir1_amid}: {len(unique_files_dir1)}')
print(f'Number of unique files in {dir2_gna}: {len(unique_files_dir2)}')

Total files in /data/home/mrichte3/RNASeq/amide/step5: 11187
Total files in /data/home/mrichte3/RNASeq/gna/step5: 11187
Number of common files: 11187
Number of unique files in /data/home/mrichte3/RNASeq/amide/step5: 0
Number of unique files in /data/home/mrichte3/RNASeq/gna/step5: 0


In [22]:
import os
import shutil

dir1_amid = '/data/home/mrichte3/RNASeq/amide/step5'
dir2_gna = '/data/home/mrichte3/RNASeq/gna/step5'
dir1_amid_orig = '/data/home/mrichte3/RNASeq/amide/original_data2'
dir2_gna_orig = '/data/home/mrichte3/RNASeq/gna/original_data2'

files_dir1 = set(os.listdir(dir1_amid))
files_dir2 = set(os.listdir(dir2_gna))

common_files = files_dir1.intersection(files_dir2)
unique_files_dir1 = files_dir1 - common_files
unique_files_dir2 = files_dir2 - common_files

for file in unique_files_dir1:
    source_path = os.path.join(dir1_amid, file)
    if os.path.isfile(source_path):
        shutil.move(source_path, dir1_amid_orig)

for file in unique_files_dir2:
    source_path = os.path.join(dir2_gna, file)
    if os.path.isfile(source_path):
        shutil.move(source_path, dir2_gna_orig)

In [24]:
# moves wrong file set
import os
import shutil

dir1_amid = '/data/home/mrichte3/RNASeq/amide/step5'
unmod_dir = '/data/home/mrichte3/RNASeq/unmod'
dir1_amid_orig = '/data/home/mrichte3/RNASeq/unmod/original_data'

files_dir1 = set(os.listdir(dir1_amid))
basenames = {os.path.splitext(file)[0] for file in files_dir1}

unmod_files = set(os.listdir(unmod_dir))
unique_unmod = {file for file in unmod_files if os.path.splitext(file)[0] in basenames}

for file in unique_unmod:
    source_path = os.path.join(unmod_dir, file)
    if os.path.isfile(source_path) and file.endswith('.pdb'):
        shutil.move(source_path, dir1_amid_orig)

In [18]:
#STORING ORIGINAL DATA
import os

dir1 = '/data/home/mrichte3/RNASeq/amide2/step5'
dir2 = '/data/home/mrichte3/RNASeq/gna2/step5'

files_dir1 = set(os.listdir(dir1))
files_dir2 = set(os.listdir(dir2))

print(f'Total files in {dir1}: {len(files_dir1)}')
print(f'Total files in {dir2}: {len(files_dir2)}')

common_files = files_dir1.intersection(files_dir2)
common_file_count = len(common_files)
print(f'Number of common files: {common_file_count}')

Total files in /data/home/mrichte3/RNASeq/amide2/step5: 306
Total files in /data/home/mrichte3/RNASeq/gna2/step5: 292
Number of common files: 273


In [19]:
#STORING ORIGINAL DATA
import os
import shutil

dir1_amid = '/data/home/mrichte3/RNASeq/amide/step5'
dir2_gna = '/data/home/mrichte3/RNASeq/gna/step5'
dir1_amid_orig = '/data/home/mrichte3/RNASeq/amide/original_data'
dir2_gna_orig = '/data/home/mrichte3/RNASeq/gna/original_data'

moved_count = 0
not_moved_count = 0

for file in common_files:
    source_path_amid = os.path.join(dir1_amid, file)
    source_path_gna = os.path.join(dir2_gna, file)
    
    if os.path.isfile(source_path_amid):
        shutil.move(source_path_amid, dir1_amid_orig)
        moved_count += 1
    else:
        not_moved_count += 1
        print(f'File not moved from amide: {file}')
        
    if os.path.isfile(source_path_gna):
        shutil.move(source_path_gna, dir2_gna_orig)
        moved_count += 1
    else:
        not_moved_count += 1
        print(f'File not moved from gna: {file}')

print(f'Total files moved: {moved_count}')
print(f'Total files not moved: {not_moved_count}')

File not moved from amide: ENSG00000116962.gro
File not moved from gna: ENSG00000116962.gro
File not moved from amide: ENSG00000118515.gro
File not moved from gna: ENSG00000164171.gro
File not moved from amide: ENSG00000182512.gro
File not moved from gna: ENSG00000182512.gro
File not moved from amide: ENSG00000277972.gro
File not moved from gna: ENSG00000204922.gro
File not moved from amide: ENSG00000126698.gro
File not moved from gna: ENSG00000126698.gro
File not moved from amide: ENSG00000176046.gro
File not moved from amide: ENSG00000279520.gro
File not moved from gna: ENSG00000279520.gro
File not moved from amide: ENSG00000182973.gro
File not moved from amide: ENSG00000156011.gro
File not moved from gna: ENSG00000156011.gro
File not moved from amide: ENSG00000054793.gro
File not moved from gna: ENSG00000054793.gro
File not moved from amide: ENSG00000269729.gro
File not moved from gna: ENSG00000269729.gro
File not moved from amide: ENSG00000005073.gro
File not moved from gna: ENSG00

In [ ]:
get_ipython().run_line_magic('run', f'graphs_xyz_rna.py')

In [8]:
########## memory flush
import gc

def try_memory_flush():
    try:
        large_data = bytearray(1024 * 1024 * 375000)  # Allocate 301200 MB 
        del large_data
        gc.collect() 
        print("Memory flush attempt complete.")
    except MemoryError:
        print("Memory flush failed: Not enough available memory.")

# Call the function after your PyTorch code
try_memory_flush()

Memory flush attempt complete.


In [3]:
get_ipython().run_line_magic('run', f'graphs_xyz.py')

Processing files:   0%|          | 0/11187 [00:00<?, ?it/s]

results rcvd


In [9]:
get_ipython().run_line_magic('run', f'graphs_distances.py')

Processing files:   0%|          | 0/11186 [00:00<?, ?it/s]

results rcvd


In [4]:
get_ipython().run_line_magic('run', f'graphs_dataonly.py')

Processing files:   0%|          | 0/1 [00:00<?, ?it/s]

Residue 22: Ref Coords None, Comp Coords [[ 5.973713 16.883533 39.150906]], Comp File /data/home/mrichte3/RNASeq/gna/step5/ENSG00000277258.gro
Residue 23: Ref Coords [[12.243 16.199 39.723]], Comp Coords [[ 9.419699 15.536837 38.39276 ]], Comp File /data/home/mrichte3/RNASeq/gna/step5/ENSG00000277258.gro
Residue 24: Ref Coords [[14.268 17.842 37.563]], Comp Coords [[11.824378 17.784798 36.562714]], Comp File /data/home/mrichte3/RNASeq/gna/step5/ENSG00000277258.gro
Residue 25: Ref Coords [[16.016 19.884 36.317]], Comp Coords [[13.857608 19.973919 35.376358]], Comp File /data/home/mrichte3/RNASeq/gna/step5/ENSG00000277258.gro
Residue 26: Ref Coords [[17.193 21.562 33.995]], Comp Coords [[15.672372 22.252754 33.75524 ]], Comp File /data/home/mrichte3/RNASeq/gna/step5/ENSG00000277258.gro
results rcvd


In [4]:
import pandas as pd
import numpy as np

normalized_rawcounts = pd.read_csv('/data/home/mrichte3/RNASeq/normalized_rawcounts.csv')
gene_alignments = pd.read_csv('/data/home/mrichte3/RNASeq/gene_alignments3.csv')

mean_nt = normalized_rawcounts['mean_nt']
mean_unmod = normalized_rawcounts['mean_unmod']
mean_amide = normalized_rawcounts['mean_amide']
mean_gna = normalized_rawcounts['mean_gna']

log2FC = pd.DataFrame({
    'ensembl_id': normalized_rawcounts.iloc[:, 0],
    'log2FC_unmod': np.log2(mean_unmod / mean_nt),
    'log2FC_amide': np.log2(mean_amide / mean_nt),
    'log2FC_gna': np.log2(mean_gna / mean_nt)
})

merged = pd.merge(gene_alignments, log2FC, on='ensembl_id', how='inner')
print(merged.head())
print(merged.shape)
merged.to_csv('/data/home/mrichte3/RNASeq/gene_alignments4.csv', index=False)

        ensembl_id           gene_aligned             target_rna  \
0  ENSG00000000003  CCATAGGAATATGAATTTCCA  CCAUAGGAAUAUGAAUUUCCA   
1  ENSG00000000419  TATATATAATATGAATATATA  UAUAUAUAAUAUGAAUAUAUA   
2  ENSG00000000457  AGGACCTAATATGCAGGGAAA  AGGACCUAAUAUGCAGGGAAA   
3  ENSG00000000460  GTCTCCTAATATGATGCACCA  GUCUCCUAAUAUGAUGCACCA   
4  ENSG00000000971  ATCTCTAATATGAGTGTTTAT  AUCUCUAAUAUGAGUGUUUAU   

   log2FC_unmod  log2FC_amide  log2FC_gna  
0     -0.255741     -0.157113   -0.136412  
1      0.201731      0.110902    0.191055  
2      0.451815     -0.090951    0.113281  
3      0.092613     -0.079419   -0.018369  
4      0.433709      0.270436    0.182343  
(11296, 6)


/data/home/mrichte3/miniconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [5]:
min_max_fc = merged[['log2FC_unmod', 'log2FC_amide', 'log2FC_gna']].agg(['min', 'max'])
print(min_max_fc)

     log2FC_unmod  log2FC_amide  log2FC_gna
min     -3.077113     -2.750446   -2.103059
max      7.406055      1.983679    5.358613


In [ ]:
# CHECKING PASSENGER STRAND LOADING
import pandas as pd
import numpy as np

# Step 1: Read and preprocess the raw count data
df_raw = pd.read_csv('/data/home/mrichte3/RNASeq/6048D_rawCounts.txt', sep='\t', index_col=0)

columns_of_interest = [
    'MR10_unmod_1', 'MR11_unmod_2', 'MR12_unmod_3',
    'MR1_NT_1', 'MR2_NT_2', 'MR3_NT_3',
    'MR4_Amide3_1', 'MR5_Amide3_2', 'MR6_Amide3_3',
    'MR7_GNA7_1', 'MR8_GNA7_2', 'MR9_GNA7_3'
]
df_counts = df_raw[columns_of_interest]

# Step 2: Normalize the counts
total_counts = df_counts.sum(axis=0)
scaling_factors = total_counts / total_counts.mean()
df_normalized = df_counts.div(scaling_factors, axis=1)

# Step 3: Calculate mean counts for each group
nt_cols = ['MR1_NT_1', 'MR2_NT_2', 'MR3_NT_3']
df_normalized['mean_nt'] = df_normalized[nt_cols].mean(axis=1)

unmod_cols = ['MR10_unmod_1', 'MR11_unmod_2', 'MR12_unmod_3']
amide_cols = ['MR4_Amide3_1', 'MR5_Amide3_2', 'MR6_Amide3_3']
gna_cols = ['MR7_GNA7_1', 'MR8_GNA7_2', 'MR9_GNA7_3']

df_normalized['mean_unmod'] = df_normalized[unmod_cols].mean(axis=1)
df_normalized['mean_amide'] = df_normalized[amide_cols].mean(axis=1)
df_normalized['mean_gna'] = df_normalized[gna_cols].mean(axis=1)

# Step 4: Print the dataframe
pd.set_option('display.max_colwidth', None)
# print(df_normalized)
# df_normalized.to_csv('normalized_rawcounts.csv', index=True)

df = df_normalized
df['unmod_avg'] = df[unmod_cols].mean(axis=1)
df['nt_avg'] = df[nt_cols].mean(axis=1)
df['amide_avg'] = df[amide_cols].mean(axis=1)
df['gna_avg'] = df[gna_cols].mean(axis=1)

df_filtered = df[(df[['unmod_avg', 'nt_avg', 'amide_avg', 'gna_avg']] >= 50).any(axis=1)]

df_filtered = df_filtered.drop(columns=['unmod_avg', 'nt_avg', 'amide_avg', 'gna_avg'])
print(df_filtered.shape)
# print(df_filtered.head(1))

df_log2 = df_filtered.copy()
df_log2['log2FC_amide_vs_nt'] = np.log2(df_log2['mean_amide'] / df_log2['mean_nt'])
df_log2['log2FC_gna_vs_nt'] = np.log2(df_log2['mean_gna'] / df_log2['mean_nt'])
df_log2['log2FC_unmod_vs_nt'] = np.log2(df_log2['mean_unmod'] / df_log2['mean_nt'])

print(df_log2.shape)
# print(df_log2.head(1))

In [23]:
# CHECKING PASSENGER STRAND LOADING
# with open('test_alignment/md0/pstrand_loading2.txt') as f:
with open('pstrand_loading2.txt') as f:
    files = {line.split(',')[0].split(': ')[1] for line in f}
unique_files = list(files)
print(unique_files[:5])
print(len(unique_files))
cif_ids = [file.split('/')[-1].replace('.cif', '') for file in unique_files]
matching_ids = df_log2[df_log2.index.isin(cif_ids)]
print(len(matching_ids))
# print(matching_ids.head(1))

filtered_matching_ids = matching_ids[(matching_ids['log2FC_amide_vs_nt'].abs() > 0.5) | 
                                      (matching_ids['log2FC_gna_vs_nt'].abs() > 0.5) | 
                                      (matching_ids['log2FC_unmod_vs_nt'].abs() > 0.5)]
print(filtered_matching_ids[['log2FC_amide_vs_nt', 'log2FC_gna_vs_nt', 'log2FC_unmod_vs_nt']])
print(len(filtered_matching_ids))

['/data/home/mrichte3/RNASeq/output_cifs/ENSG00000185219.cif', '/data/home/mrichte3/RNASeq/output_cifs/ENSG00000088836.cif', '/data/home/mrichte3/RNASeq/output_cifs/ENSG00000169660.cif', '/data/home/mrichte3/RNASeq/output_cifs/ENSG00000135778.cif', '/data/home/mrichte3/RNASeq/output_cifs/ENSG00000105202.cif']
113
113
                 log2FC_amide_vs_nt  log2FC_gna_vs_nt  log2FC_unmod_vs_nt
ENSG00000029725            0.536764          0.489157            0.458109
ENSG00000077585           -0.354087         -0.385688           -0.929459
ENSG00000103160           -0.572955         -0.214556           -0.247930
ENSG00000106688            0.220323          0.410998            0.768770
ENSG00000142871           -0.113086         -0.353355           -0.622692
ENSG00000146476            0.300911          0.930359            0.877088
ENSG00000153560           -0.178618         -0.289102           -0.863423
ENSG00000165072           -0.059209         -0.449717           -0.572315
ENSG00000172000

In [12]:
## CHECK REMADE CIFS FOR PSTRAND LOADED. STILL LOADS

from pymol import cmd
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

# amide_dir = '/data/home/mrichte3/RNASeq/amide/step5'
# gna_dir = '/data/home/mrichte3/RNASeq/gna/step5'
# files = []
# for directory in [amide_dir, gna_dir]:
#     for filename in os.listdir(directory):
#         if filename.endswith(('.cif', '.gro', '.pdb')):
#             files.append(os.path.join(directory, filename))

cif_dir = '/data/home/mrichte3/RNASeq/pstrand_redo'
files = []
for directory in [cif_dir]:
    for filename in os.listdir(directory):
        if filename.endswith(('.cif', '.gro', '.pdb')):
            files.append(os.path.join(directory, filename))

def check_file(file):
    cmd.reinitialize()
    cmd.load(file)
    cmd.select("first_rna_base", f"byres (first (byres (name C3') and resi 1))")
    cmd.select('not_C3_ref', 'not byres name C3\'')
    cmd.select("resi_859", "not_C3_ref and resi 859")
    cmd.select("resi_812", "not_C3_ref and resi 812")
    cmd.select("resi_815", "not_C3_ref and resi 815")
    cmd.select("resi_566", "not_C3_ref and resi 566")
    results = []
    for resi in [859, 812, 815, 566]:
        dist_value = cmd.distance(f"dist_first_rna_base_resi_{resi}", "first_rna_base", f"resi_{resi}")
        if dist_value > 21: ###19.5 was the highest I found, setting to 21 but 20 should be good
            results.append(f"File: {file}, Resi: {resi}, Distance: {dist_value}")
    return results

with ProcessPoolExecutor(max_workers=48) as executor:
    all_results = list(tqdm(executor.map(check_file, files), total=len(files)))
    # with open('pstrand_loading2.txt', 'w') as f:
    #     for results in all_results:
    #         for result in results:
    #             f.write(result + '\n')
    print(all_results)


100%|██████████| 1/1 [00:00<00:00,  7.71it/s]

[['File: /data/home/mrichte3/RNASeq/pstrand_redo/ENSG00000196591.cif, Resi: 859, Distance: 53.2643928527832', 'File: /data/home/mrichte3/RNASeq/pstrand_redo/ENSG00000196591.cif, Resi: 812, Distance: 47.596153259277344', 'File: /data/home/mrichte3/RNASeq/pstrand_redo/ENSG00000196591.cif, Resi: 815, Distance: 43.435081481933594', 'File: /data/home/mrichte3/RNASeq/pstrand_redo/ENSG00000196591.cif, Resi: 566, Distance: 55.6408576965332']]


In [ ]:
## CHECK ORIGINAL CIF FILES FOR PSTRAND LOADED

from pymol import cmd
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

# amide_dir = '/data/home/mrichte3/RNASeq/amide/step5'
# gna_dir = '/data/home/mrichte3/RNASeq/gna/step5'
# files = []
# for directory in [amide_dir, gna_dir]:
#     for filename in os.listdir(directory):
#         if filename.endswith(('.cif', '.gro', '.pdb')):
#             files.append(os.path.join(directory, filename))

cif_dir = '/data/home/mrichte3/RNASeq/output_cifs'
files = []
for directory in [cif_dir]:
    for filename in os.listdir(directory):
        if filename.endswith(('.cif', '.gro', '.pdb')):
            files.append(os.path.join(directory, filename))

def check_file(file):
    cmd.reinitialize()
    cmd.load(file)
    cmd.select("first_rna_base", f"byres (first (byres (name C3') and resi 1))")
    cmd.select('not_C3_ref', 'not byres name C3\'')
    cmd.select("resi_859", "not_C3_ref and resi 859")
    cmd.select("resi_812", "not_C3_ref and resi 812")
    cmd.select("resi_815", "not_C3_ref and resi 815")
    cmd.select("resi_566", "not_C3_ref and resi 566")
    results = []
    for resi in [859, 812, 815, 566]:
        dist_value = cmd.distance(f"dist_first_rna_base_resi_{resi}", "first_rna_base", f"resi_{resi}")
        if dist_value > 21: ###19.5 was the highest I found, setting to 21 but 20 should be good
            results.append(f"File: {file}, Resi: {resi}, Distance: {dist_value}")
    return results

with ProcessPoolExecutor(max_workers=48) as executor:
    all_results = list(tqdm(executor.map(check_file, files), total=len(files)))
    with open('pstrand_loading2.txt', 'w') as f:
        for results in all_results:
            for result in results:
                f.write(result + '\n')


In [ ]:
## PSTRAND LOADING: CHECK GNA AND AMIDE FILES for first resi in guide position

from pymol import cmd
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

amide_dir = '/data/home/mrichte3/RNASeq/amide/step5'
gna_dir = '/data/home/mrichte3/RNASeq/gna/step5'
files = []
for directory in [amide_dir, gna_dir]:
    for filename in os.listdir(directory):
        if filename.endswith(('.cif', '.gro', '.pdb')):
            files.append(os.path.join(directory, filename))

def check_file(file):
    cmd.reinitialize()
    cmd.load(file)
    cmd.select("first_rna_base", f"byres (first (byres (name C3') and resi 1))")
    cmd.select('not_C3_ref', 'not byres name C3\'')
    cmd.select("resi_859", "not_C3_ref and resi 859")
    cmd.select("resi_812", "not_C3_ref and resi 812")
    cmd.select("resi_815", "not_C3_ref and resi 815")
    cmd.select("resi_566", "not_C3_ref and resi 566")
    results = []
    for resi in [859, 812, 815, 566]:
        dist_value = cmd.distance(f"dist_first_rna_base_resi_{resi}", "first_rna_base", f"resi_{resi}")
        if dist_value > 15:
            results.append(f"File: {file}, Resi: {resi}, Distance: {dist_value}")
    return results

with ProcessPoolExecutor(max_workers=48) as executor:
    all_results = list(tqdm(executor.map(check_file, files), total=len(files)))
    with open('files_to_check.txt', 'w') as f:
        for results in all_results:
            for result in results:
                f.write(result + '\n')
# with ProcessPoolExecutor(max_workers=48) as executor:
#     all_results = list(tqdm(executor.map(check_file, files[:1000]), total=100))
#     for results in all_results:
#         for result in results:
#             print(result)
# for file in files[:5]:
#     cmd.reinitialize()
#     cmd.load(file)
#     cmd.select("first_rna_base", f"byres (first (byres (name C3') and resi 1))")
#     cmd.select('not_C3_ref', 'not byres name C3\'')
#     cmd.select("resi_859", "not_C3_ref and resi 859")
#     cmd.select("resi_812", "not_C3_ref and resi 812")
#     cmd.select("resi_815", "not_C3_ref and resi 815")
#     cmd.select("resi_566", "not_C3_ref and resi 566")
#     print(file)
#     for resi in [859, 812, 815, 566]:
#         dist_value = cmd.distance(f"dist_first_rna_base_resi_{resi}", f"first_rna_base", f"resi_{resi}")
#         print(f"Distance between first_rna_base and resi {resi}: {dist_value}")

In [1]:
## CHECK ALL OUTPUT DATA, ENSURE THAT MODIFICATION IS PRESENT (GNA OR AMIDE)

from pymol import cmd
import os
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

amide_dir = '/data/home/mrichte3/RNASeq/amide/step5'
gna_dir = '/data/home/mrichte3/RNASeq/gna/step5'
amide_files = []
gna_files = []
for filename in os.listdir(amide_dir):
    if filename.endswith(('.cif', '.gro', '.pdb')):
        amide_files.append(os.path.join(amide_dir, filename))
for filename in os.listdir(gna_dir):
    if filename.endswith(('.cif', '.gro', '.pdb')):
        gna_files.append(os.path.join(gna_dir, filename))




def check_gna_file(gna_file):
    cmd.reinitialize()
    cmd.load(gna_file)
    cmd.select("sel_gna", "byres (first (name C3G and resi 7 and resn GNAU))")
    found_gnau = cmd.count_atoms("sel_gna") > 0
    if not found_gnau:
        print(f"Residue GNAU not found in {gna_file}")

print("checking gna files")
with ProcessPoolExecutor(max_workers=48) as executor:
    list(tqdm(executor.map(check_gna_file, gna_files), total=len(gna_files)))
print("gna files completed")

def check_amide_file(amide_file):
    cmd.reinitialize()
    cmd.load(amide_file)
    cmd.select("sel_amide", "byres (first (name C3' and resi 3 and resn R3))")
    found_amide = cmd.count_atoms("sel_amide") > 0
    if not found_amide:
        print(f"Residue R3 not found in {amide_file}")

print(f"Starting check for amide files")
with ProcessPoolExecutor(max_workers=48) as executor:
    list(tqdm(executor.map(check_amide_file, amide_files), total=len(amide_files)))

checking gna files


100%|██████████| 6117/6117 [03:20<00:00, 30.58it/s]

gna files completed
Starting check for amide files



100%|██████████| 6114/6114 [04:28<00:00, 22.79it/s]


In [ ]:
###TRYING 5 SUBPROCESS
import subprocess
import os
import time
from concurrent.futures import ProcessPoolExecutor

os.environ['PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/bin:' + os.environ['PATH']
if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib:' + os.environ['LD_LIBRARY_PATH']
else:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib'
os.environ['GMX_MAXBACKUP'] = '-1'
os.environ["OMP_NUM_THREADS"] = "1"
def run_command(command):
    result = subprocess.run(command, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for line in output.splitlines():
        if "warning" in line.lower() or "fatal" in line.lower():
            print(line)

def run_gromacs_process(i):
    # os.environ["OMP_NUM_THREADS"] = "1"
    mini_prefix = f"step4.0_minimization_{i}"
    command_grompp = ["gmx", "grompp", "-v", "-f", "step4.0_minimization.mdp", "-o", f"{mini_prefix}.tpr", "-c", "structure_solv_ions.gro", "-r", "structure_solv_ions.gro", "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5"]
    start_time = time.time()
    run_command(command_grompp)
    command_mdrun = ["gmx", "mdrun", "-v", "-deffnm", mini_prefix, "-ntmpi", "1"]
    run_command(command_mdrun)
    elapsed_time = time.time() - start_time
    print(f"Process {i} completed in {elapsed_time:.2f} seconds.")

with ProcessPoolExecutor(max_workers=6) as executor:
    executor.map(run_gromacs_process, range(20, 61))

In [ ]:
import subprocess
import os
import shutil
os.environ['PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/bin:' + os.environ['PATH']
if 'LD_LIBRARY_PATH' in os.environ:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib:' + os.environ['LD_LIBRARY_PATH']
else:
    os.environ['LD_LIBRARY_PATH'] = '/data/home/mrichte3/gromacs-2024.2/install/lib'
os.environ['GMX_MAXBACKUP'] = '-1'

def run_command(command, input_text=None, max_chars=100):
    result = subprocess.run(command, capture_output=True, text=True, input=input_text)
    print(result.stdout)
    print(result.stderr)
    output = result.stdout + result.stderr
    for line in output.splitlines():
        if "warning" in line.lower() or "fatal" in line.lower() or "random" in line.lower():
            print(line[:max_chars])
    
def run_gromacs_commands():
    command = ["gmx", "pdb2gmx", "-f", "pdb_with_modification.pdb", "-o", "structure_processed.gro", 
               "-p", "topol.top", "-i", "posre.itp"]
    input_text = "6\n1\n"        ############ 6 1 for custom
    run_command(command, input_text)
    command = ["gmx", "editconf", "-f", "structure_processed.gro", "-o", "structure_box.gro", "-c", "-d", "1.0", "-bt", "cubic"]
    run_command(command)
    command = ["gmx", "solvate", "-cp", "structure_box.gro", "-cs", "spc216.gro", "-o", "structure_solv.gro", "-p", "topol.top"]
    run_command(command)
    command = ["gmx", "grompp", "-f", "ions.mdp", "-c", "structure_solv.gro", "-p", "topol.top", "-o", "ions.tpr", "-maxwarn", "3"]
    run_command(command)
    command = ["gmx", "genion", "-s", "ions.tpr", "-o", "structure_solv_ions.gro", "-p", "topol.top", 
               "-pname", "NA", "-nname", "CL", "-neutral", "-conc", "0.15", "-seed", "12345"]
    input_text = "14\n"
    run_command(command, input_text)
    command = ["gmx", "make_ndx", "-f", "structure_solv_ions.gro", "-o", "index.ndx"]
    input_text = "name 19 SOLV\n1 | 12\nname 20 SOLU\nq\n"
    run_command(command, input_text)

run_gromacs_commands()
mini_prefix = "step4.0_minimization"
equi_prefix = "step4.0_minimization"
prod_prefix = "step5_production"
prod_step = "step5"
init = "structure_solv_ions"
topol_file = "topol.top"
gro_file = f"{init}.gro"

command = ["gmx", "grompp", "-v", "-f", f"{mini_prefix}.mdp", "-o", f"{mini_prefix}.tpr", 
           "-c", gro_file, "-r", gro_file, "-p", topol_file, "-n", "index.ndx", "-maxwarn", "5"]
run_command(command)
command = ["gmx", "mdrun", "-v", "-deffnm", mini_prefix, "-ntmpi", "1"]
run_command(command)

command_grompp = ["gmx", "grompp", "-f", f"{prod_prefix}.mdp", "-o", f"{prod_step}.tpr", 
                  "-c", f"{equi_prefix}.gro", "-p", "topol.top", "-n", "index.ndx", "-maxwarn", "5"]
run_command(command_grompp)

command_mdrun = ["gmx", "mdrun", "-v", "-deffnm", prod_step, "-ntmpi", "1"]
run_command(command_mdrun)




In [12]:
import subprocess

def run_command(command):
    return subprocess.run(command, capture_output=True, text=True)

base_command = ["gmx", "editconf", "-f", "step4.0_minimization_1.gro", "-o", "aligned_minimized_1.gro"]
results = []

for x in [-1, 1]:
    for y in [-1, 1]:
        for z in [-1, 1]:
            align_command = base_command + ["-align", str(x), str(y), str(z)]
            run_command(align_command)
            result = subprocess.run(["grep", "    1MET      N    1", "aligned_minimized_1.gro"], capture_output=True, text=True)
            if result.stdout:
                results.append((x, y, z, result.stdout.strip()))

print("{:<5} {:<5} {:<5} {}".format("X", "Y", "Z", "Position"))
for x, y, z, position in results:
    print("{:<5} {:<5} {:<5} {}".format(x, y, z, position))

X     Y     Z     Position
-1    -1    -1    1MET      N    1  -9.327  -6.257  -3.091
-1    -1    1     1MET      N    1  -9.217  -2.994   6.464
-1    1     -1    1MET      N    1  -9.214   2.992  -6.470
-1    1     1     1MET      N    1  -9.215   6.467   2.993
1     -1    -1    1MET      N    1   9.305  -3.070  -6.300
1     -1    1     1MET      N    1   6.383  -3.031   9.262
1     1     -1    1MET      N    1   3.030   9.261  -6.385
1     1     1     1MET      N    1   3.204   9.432   6.039


In [ ]:
get_ipython().run_line_magic('run', f'graphs_stdev2.py 0')
# get_ipython().run_line_magic('run', f'graphs_stdev2.py 0')

In [1]:
get_ipython().run_line_magic('run', f'graphs_xyz_rna.py')


Processing files:   0%|          | 0/8666 [00:00<?, ?it/s]

results rcvd


In [6]:
########## memory flush
import gc

def try_memory_flush():
    try:
        large_data = bytearray(1024 * 1024 * 375000)  # Allocate 301200 MB 
        del large_data
        gc.collect() 
        print("Memory flush attempt complete.")
    except MemoryError:
        print("Memory flush failed: Not enough available memory.")

# Call the function after your PyTorch code
try_memory_flush()

Memory flush attempt complete.


In [ ]:
# ### OLD GRAPH
# import subprocess
# import os
# import shutil
# import time
# import numpy as np
# print("graph modified to md2 dir, steps = 150, no graphing")

# for i in range(20, 301):

#     if i % 10 == 0 and i > 20:
#         igroup = int(i / 10) -1
#         print(igroup)
#         get_ipython().run_line_magic('run', f'graphs_stdev2.py {igroup}')


# print(f"Sum of relative standard deviations: {rsd_sums:.4f}")

import os
import numpy as np
import re
from io import StringIO
import sys

results = []
results_diff = []
results_rsd = []

for i in range(10, 11):
    if i % 10 == 0:
        igroup = int(i / 10) - 1
        print(igroup)
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        get_ipython().run_line_magic('run', f'graphs_dataonly.py {igroup}')
        output = sys.stdout.getvalue()
        sys.stdout = old_stdout
        print(output)
        # matches = re.findall(r'Sum of relative standard deviations: (\d+\.\d+)', output)
        matches = re.findall(r'Sum of differences between first and last pair: (\d+\.\d+)', output)
        results.extend(float(match) for match in matches)
        ############new
        matches_rsd = re.findall(r'Sum of relative standard deviations: (\d+\.\d+)', output)
        matches_diff = re.findall(r'Sum of differences between first and last pair: (\d+\.\d+)', output)
        results_diff.extend(float(match) for match in matches_diff)
        results_rsd.extend(float(match) for match in matches_rsd)
        ##############
        break


average_rsd = sum(results_rsd) / len(results_rsd) if results_rsd else None
average_diff = sum(results_diff) / len(results_diff) if results_diff else None

print(f"avg rsd: {average_rsd} n = {len(results_rsd)}")
print(f"avg differences: {average_diff} n = {len(results_diff)}")

average_rsd = sum(results) / len(results) if results else None

print(f"avg rsd: {average_rsd} n = {len(results)}")



In [26]:
# VELOCITIES
import numpy as np

def analyze_velocities(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    data = []
    for line in lines:
        if line.startswith('#') or line.startswith('@'):
            continue
        values = line.split()
        if values:  # Ensure there is data in the line
            data.append([float(v) for v in values])  # Convert to float

    data = np.array(data)  # Convert to NumPy array
    num_data_points = data.shape[0]  # Number of data points

    print(f"Total number of data points: {num_data_points}")
    print(f"Number of columns per data point: {data.shape[1] if num_data_points > 0 else 0}")

    if num_data_points > 0:
        for i in range(num_data_points):
            time = data[i, 0]  # First value is time
            velocities = data[i, 1:4]  # Next three values are X, Y, Z for the first atom
            print(f"Time: {time:.5f}, Atom 1 Velocities: X: {velocities[0]:.5f}, Y: {velocities[1]:.5f}, Z: {velocities[2]:.5f}")
            
            # Print velocities for the first three atoms
            if i < num_data_points:
                velocities_atom_2 = data[i, 4:7]  # X, Y, Z for the second atom
                velocities_atom_3 = data[i, 7:10]  # X, Y, Z for the third atom
                print(f"Atom 2 Velocities: X: {velocities_atom_2[0]:.5f}, Y: {velocities_atom_2[1]:.5f}, Z: {velocities_atom_2[2]:.5f}")
                print(f"Atom 3 Velocities: X: {velocities_atom_3[0]:.5f}, Y: {velocities_atom_3[1]:.5f}, Z: {velocities_atom_3[2]:.5f}")
            print()  # Separate output for each time point

analyze_velocities('velocities.xvg')

Total number of data points: 6
Number of columns per data point: 41119
Time: 0.00000, Atom 1 Velocities: X: 0.00000, Y: 0.00002, Z: 0.00001
Atom 2 Velocities: X: -0.00002, Y: 0.00003, Z: -0.00007
Atom 3 Velocities: X: -0.00007, Y: -0.00022, Z: -0.00001

Time: 0.20000, Atom 1 Velocities: X: -0.00892, Y: 0.04152, Z: 0.08045
Atom 2 Velocities: X: -0.15873, Y: -0.14378, Z: 0.01417
Atom 3 Velocities: X: -0.58300, Y: 0.31610, Z: -0.50006

Time: 0.40000, Atom 1 Velocities: X: 0.02436, Y: -0.51599, Z: -0.12200
Atom 2 Velocities: X: 0.15746, Y: -0.55869, Z: -0.17289
Atom 3 Velocities: X: -0.63740, Y: -0.28946, Z: -0.19090

Time: 0.60000, Atom 1 Velocities: X: 0.50561, Y: 0.24415, Z: -0.01635
Atom 2 Velocities: X: -0.46471, Y: -0.05457, Z: 0.12966
Atom 3 Velocities: X: -1.34557, Y: 0.83219, Z: -1.11792

Time: 0.80000, Atom 1 Velocities: X: -0.50945, Y: -0.13365, Z: 0.14558
Atom 2 Velocities: X: -0.66241, Y: -0.83184, Z: -0.00569
Atom 3 Velocities: X: 0.03989, Y: -0.30388, Z: 0.32187

Time: 1.000

In [ ]:
# VELOCITIES
import numpy as np

def analyze_velocities_overall(file_prefix, num_files):
    print("Time: 0.2")
    for i in range(1, num_files + 1):
        file_path = f"{dir_path}/{file_prefix}_{i}.xvg"
        with open(file_path, 'r') as file:
            lines = file.readlines()
        data = []
        for line in lines:
            if line.startswith('#') or line.startswith('@'):
                continue
            values = line.split()
            if values:
                data.append([float(v) for v in values])
        data = np.array(data)
        num_data_points = data.shape[0]
        if num_data_points > 0:
            total_velocities = np.zeros(3)
            atom_sums = np.zeros(data.shape[1] // 3 - 1)
            for j in range(num_data_points):
                time = data[j, 0]
                if time in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
                    for atom in range(atom_sums.size):
                        velocities_atom = np.abs(data[j, (atom * 3 + 1):(atom * 3 + 4)])
                        atom_sums[atom] += np.sum(velocities_atom)
                    total_sum = np.sum(atom_sums)
                    print(f"File {i}, Total Velocities (Sum) at time {time:.1f}: {total_sum:.5f}")

analyze_velocities_overall('velocities', 10)

In [52]:
# VELOCITIES
import numpy as np

def collect_velocities(file_prefix):
    file_path = f"{dir_path}/{file_prefix}_1.xvg"
    with open(file_path, 'r') as file:
        lines = file.readlines()
    data = []
    for line in lines:
        if line.startswith('#') or line.startswith('@'):
            continue
        values = line.split()
        if values:
            data.append([float(v) for v in values])
    data = np.array(data)
    num_data_points = data.shape[0]
    
    velocities = []
    if num_data_points > 0:
        for j in range(num_data_points):
            time = data[j, 0]
            if time == 0.0:
                velocities_atom_1 = data[j, 1:4]  # X, Y, Z for the first atom
                velocities_atom_2 = data[j, 4:7]  # X, Y, Z for the second atom
                velocities_atom_3 = data[j, 7:10]  # X, Y, Z for the third atom
                velocities.append((velocities_atom_1, velocities_atom_2, velocities_atom_3))
    
    return velocities

def append_velocities_to_positions(position_file, velocity_file_prefix):
    with open(position_file, 'r') as file:
        position_lines = file.readlines()
    
    velocities = collect_velocities(velocity_file_prefix)

    for idx, line in enumerate(position_lines[2:5]):  # Adjust for specific atoms
        if idx < len(velocities):
            atom_data = velocities[idx]
            print(line.strip(), f"{atom_data[0][0]:.3f} {atom_data[0][1]:.3f} {atom_data[0][2]:.3f}")
            print(line.strip().replace("N", "H1") if idx == 0 else line.strip().replace("H1", "H2"), 
                  f"{atom_data[1][0]:.3f} {atom_data[1][1]:.3f} {atom_data[1][2]:.3f}")
            print(line.strip().replace("H2", "H2"), 
                  f"{atom_data[2][0]:.3f} {atom_data[2][1]:.3f} {atom_data[2][2]:.3f}")

append_velocities_to_positions('step4.0_minimization.gro', 'velocities')

1MET      N    1   3.338   9.441   6.248 -0.000 0.000 0.000
1MET      H1    1   3.338   9.441   6.248 0.000 -0.000 0.000
1MET      N    1   3.338   9.441   6.248 -0.000 -0.000 -0.000
